# Классы и ООП (часть 2)

In [120]:
import re


class UnsupportedLanguageError(Exception):
    def __init__(self, allowed_languages: list[str]):
        self.allowed_languages = allowed_languages

    def __str__(self):
        joined = ", ".join(self.allowed_languages)
        return f"Выберите поддерживаемый язык, список поддерживаемых языков: {joined}"


class Word:

    allowed_languages = ['русский', 'английский']

    def __init__(self, glossed: str, gloss: str, language: str):
        self._check_language(language)

        self.glossed = glossed
        self.gloss = gloss
        self.language = language


    def __len__(self) -> int:
        count_letters = 0
        for letter in self.glossed:
            if letter.isalpha():
                count_letters += 1
        return count_letters
    
    def __str__(self) -> str:
        return f'{self.glossed}\n{self.gloss}'
    
    def get_morphemes(self, delimiters: str) -> list[str]:
        if "-" in delimiters:
            delimiters = delimiters.replace("-", "") + "-"
        
        return re.split(rf"[{delimiters}]+", self.glossed.strip(delimiters))
    
    def count_morphemes(self, delimiters: str) -> int:
        return len(self.get_morphemes(delimiters))

    
    def _check_language(self, language: str):
        if language in self.allowed_languages:
            return
        
        user_prompt = input(f"Языка {language} нет в списке разрешенных, добавить? (да/нет)")

        if user_prompt == "да":
            self.allowed_languages.append(language)
            return
        raise UnsupportedLanguageError(allowed_languages=self.allowed_languages)

    __repr__ = __str__

### <mark>Мы закончили первый семинар тут</mark>

Magic-методы и прочее. А как вообще работает `len()` под капотом?

In [4]:
len(Word('мам-а==аааааааааа.', 'mother-NOM.SG=а', "русский"))

14

In [ ]:
from typing import Any


def my_len(obj: Any):
    if hasattr(obj, '__len__') and callable(obj.__len__):
        return obj.__len__()
    raise TypeError(f'object of type {type(obj).__name__} has no len()')

In [125]:
my_len(Word('мам-ы', 'mother-GEN', 'русский'))

4

In [126]:
len(Word('мам-ы', 'mother-GEN', 'русский'))

4

## `@staticmethod`

Иногда метод объекта никак не использует его атрибуты (не обращается к `self`), а просто выполняет независимую логику. В таком случае такой метод называется статическим. Мы можем использовать на нём специальный декоратор `@staticmethod`, и тогда мы помимо прочего сможем вызывать его, даже не создавая объект.

Давайте напишем такой статический метод, который будет переводить кириллическую строку в латиническую и наоборот. Назовём метод `transliterate`. Он будет принимать строку и таргетное значение "кодировки" (`lat` или `cyr`).

P.S. Давайте считать, что е, ё, ю, я всегда переводятся в j + eoua.

In [ ]:
from functools import lru_cache


@lru_cache(10)
def fibonacci(n: int) -> int:
    if n == 1 or n == 2:
        return 1
    return fibonacci(n - 1) + fibonacci(n - 2)

In [143]:
import re
from typing import Literal


class UnsupportedLanguageError(Exception):
    def __init__(self, allowed_languages: list[str]):
        self.allowed_languages = allowed_languages

    def __str__(self):
        joined = ", ".join(self.allowed_languages)
        return f"Выберите поддерживаемый язык, список поддерживаемых языков: {joined}"


class Word:

    allowed_languages = ['русский', 'английский']

    def __init__(self, glossed: str, gloss: str, language: str):
        self._check_language(language)

        self.glossed = glossed
        self.gloss = gloss
        self.language = language


    def __len__(self) -> int:
        count_letters = 0
        for letter in self.glossed:
            if letter.isalpha():
                count_letters += 1
        return count_letters
    
    def __str__(self) -> str:
        return f'{self.glossed}\n{self.gloss}'
    
    def get_morphemes(self, delimiters: str) -> list[str]:
        if "-" in delimiters:
            delimiters = delimiters.replace("-", "") + "-"
        
        return re.split(rf"[{delimiters}]+", self.glossed.strip(delimiters))
    
    def count_morphemes(self, delimiters: str) -> int:
        return len(self.get_morphemes(delimiters))

    
    def _check_language(self, language: str):
        if language in self.allowed_languages:
            return
        
        user_prompt = input(f"Языка {language} нет в списке разрешенных, добавить? (да/нет)")

        if user_prompt == "да":
            self.allowed_languages.append(language)
            return
        raise UnsupportedLanguageError(allowed_languages=self.allowed_languages)
    

    @staticmethod
    def transliterate(word: str, target: Literal['lat', 'cyr']) -> str:
        cyr = "а б в г д е ё ж з и к л м н о п р с т у ф х ц ч ш щ ъ ы ь э ю я".split()
        lat = "a b v g d je jo j z i k l m n o p r s t u f h c ch sh sch '' y ' e ju ja".split()
        lat2cyr = dict(zip(lat, cyr))
        cyr2lat = dict(zip(cyr, lat))

        target_dict: dict[str, str]
        if target == 'cyr':
            target_dict = lat2cyr
        elif target == 'lat':
            target_dict = cyr2lat
        else:
            raise ValueError(f"Unsupported target: {target}")
        
        # target_dict.__getitem__(...) равносильно target_dict[...]
        return "".join(map(target_dict.__getitem__, word.split()))
        
        # result = []
        # for letter in word.split():
        #     result.append(target_dict[letter])
        # return "".join(result)

    def __getitem__(self, item: Any) -> int:
        return self.glossed[item]


    def __str__(self) -> str:
        return f'{self.glossed}\n{self.gloss}\n{self.transliterate(self.glossed)}'

    __repr__ = __str__

In [141]:
"abc" + "def"

'abcdef'

In [144]:
w = Word('мам-ы', 'mother-GEN', 'русский')
w[2]

'м'

In [142]:
Word.transliterate('l i n g v i s t i k a', 'cyr')

'лингвистика'

## `@classmethod`

Допустим, мы хотим преобразовать "сырые" сточки вида `мам-а mother-NOM.SG` в экземпляр класса `Word`

In [71]:
class Word:

    def __init__(self, glossed, gloss):
        self.glossed = glossed
        self.gloss = gloss

    def __str__(self):
        return f'{self.glossed}\n{self.gloss}'

    __repr__ = __str__

def from_raw(raw_word: str) -> Word:
    glossed, gloss = raw_word.split()
    return Word(glossed, gloss)

In [72]:
type(from_raw('мам-ы mother-GEN.SG'))

__main__.Word

А что если мы хотим вызывать это так: `Word.from_raw(мам-а mother-NOM.SG)`?

In [158]:
import re
from typing import Literal, Self


class UnsupportedLanguageError(Exception):
    def __init__(self, allowed_languages: list[str]):
        self.allowed_languages = allowed_languages

    def __str__(self):
        joined = ", ".join(self.allowed_languages)
        return f"Выберите поддерживаемый язык, список поддерживаемых языков: {joined}"


class Word:

    allowed_languages = ['русский', 'английский']

    def __init__(self, glossed: str, gloss: str, language: str):
        self._check_language(language)

        self.glossed = glossed
        self.gloss = gloss
        self.language = language


    def __len__(self) -> int:
        count_letters = 0
        for letter in self.glossed:
            if letter.isalpha():
                count_letters += 1
        return count_letters
    
    def __str__(self) -> str:
        return f'{self.glossed}\n{self.gloss}'
    
    def get_morphemes(self, delimiters: str) -> list[str]:
        if "-" in delimiters:
            delimiters = delimiters.replace("-", "") + "-"
        
        return re.split(rf"[{delimiters}]+", self.glossed.strip(delimiters))
    
    def count_morphemes(self, delimiters: str) -> int:
        return len(self.get_morphemes(delimiters))

    
    def _check_language(self, language: str):
        if language in self.allowed_languages:
            return
        
        user_prompt = input(f"Языка {language} нет в списке разрешенных, добавить? (да/нет)")

        if user_prompt == "да":
            self.allowed_languages.append(language)
            return
        raise UnsupportedLanguageError(allowed_languages=self.allowed_languages)
    

    @staticmethod
    def transliterate(word: str, target: Literal['lat', 'cyr']) -> str:
        cyr = "а б в г д е ё ж з и к л м н о п р с т у ф х ц ч ш щ ъ ы ь э ю я".split()
        lat = "a b v g d je jo j z i k l m n o p r s t u f h c ch sh sch '' y ' e ju ja".split()
        lat2cyr = dict(zip(lat, cyr))
        cyr2lat = dict(zip(cyr, lat))

        target_dict: dict[str, str]
        if target == 'cyr':
            target_dict = lat2cyr
        elif target == 'lat':
            target_dict = cyr2lat
        else:
            raise ValueError(f"Unsupported target: {target}")
        
        # target_dict.__getitem__(...) равносильно target_dict[...]
        return "".join(map(target_dict.__getitem__, word.split()))
        
        # result = []
        # for letter in word.split():
        #     result.append(target_dict[letter])
        # return "".join(result)

    def __getitem__(self, item: Any) -> int:
        return self.glossed[item]


    def __str__(self) -> str:
        return f'{self.glossed}\n{self.gloss}'
    
    @classmethod
    def from_raw(cls, raw_word: str, language: str) -> Self:
        glossed, gloss = raw_word.split()
        return cls(glossed, gloss, language)

    __repr__ = __str__

In [159]:
Word.from_raw('мам-ы mother-GEN.SG', 'русский')

мам-ы
mother-GEN.SG

На самом деле, метод класса, в отличие от статика, может менять **только** состояние самого класса (не его экземпляров). Первым аргументом всегда идет сам класс, обычно в виде `cls`. Их можно наследовать и переопределять (см. дальше).

## Наследование, `super()`


Одним из главных механизмов ООП является наследование. Класс может отнаследоваться от другого класса, тем самым получая доступ к его методам и атрибутам. Класс-ребёнок может добавлять свои новые методы и переопределять методы класса-родителя, но так же может использовать и все методы, определённые в классе родителе.

Родительский класс в питоне указывается в скобках после названия класса во время объявления: `class Children(Parent)`. Причём один класс может быть ребёнком сразу нескольких классов. В таком случае они перечисляются через запятую.

In [167]:
class Noun(Word):

    def __init__(self, glossed: str, gloss: str, language: str):
        super().__init__(glossed, gloss, language)
        self.pos = 'NOUN'
    
    def __str__(self):
        return super().__str__() + f'\n{self.pos}'

    __repr__ = __str__

In [168]:
mother_noun = Noun('мам-а', 'mother-NOM.SG', 'русский')
mother_noun

мам-а
mother-NOM.SG
NOUN

In [169]:
mother_noun.pos

'NOUN'

In [170]:
print(Noun.from_raw('мам-а mother-NOM.SG', 'русский'))

мам-а
mother-NOM.SG
NOUN


In [171]:
m = Noun.from_raw('мам-а mother-NOM.SG', 'русский')
m.pos

'NOUN'

Стоит обратить особое внимание на оператор `super()`. Все методы, вызывающиеся от `super()`, будут вызваны от родительского класса (классов) объекта. Таким образом, когда мы вызываем `super().__init__(name)` в классах-детях мы выполняем код из метода `__init__` родительского класса.

Мы уже встречались когда-то с функцией `isinstance`, которая проверяла тип данных объекта.

In [173]:
print(isinstance('sdg', str))
print(type('sdg') == str)

True
True


Давайте посмотрим, что будет с нашими классами и функцией `isinstance` vs функцией `type`:

In [174]:
print(
    type(mother_noun) == Noun,
    type(mother_noun) == Word,
    isinstance(mother_noun, Noun),
    isinstance(mother_noun, Word)
    )

True False True True


In [ ]:
Noun.__mro__  # method resolution error

# return obj.__class__.__mro__[0]

(__main__.Noun, __main__.Word, object)

# Отвлечённое задание

У списка есть метод `remove`, который удаляет первое вхождение некоторого элемента. Напишите такой класс `CustomList`, у которого будут все методы обычного списка, но будет ещё один метод `remove_all`, который будет удалять все вхождения элемента, который он получает на вход.

In [ ]:
lst = [1, 2, 3, 2]
lst.remove(2)
print(lst)

[3, 4, 1, 2, 2]

[1, 3, 2]


In [183]:
class CustomList(list):
    def remove_all(self, value):
        pos = 0
        to_remove = 0
        for i in range(len(self)):
            if self[i] != value:
                self[pos] = self[i]
                pos += 1
            else:
                to_remove += 1

        for _ in range(to_remove):
            self.pop()

In [184]:
lst = CustomList((1, 5, 3, 2, 3))
lst.remove_all(3)

lst

[1, 5, 2]

# Итоговая задача

Скачайте [файл](https://raw.githubusercontent.com/vantral/files/main/sentence.txt) с разбором одного предложения на эвенском языке.

Считайте, что глоссы `pl` и `loc` $-$ это именные глоссы, а глоссы `cond`, `pst`, `nfut` и `cvb` $-$ это глагольные глоссы. Допишите класс `Noun` так, чтобы он хранил информацию о числе имени: множественное если `pl` есть, единственное $-$ если `pl` нет.

Напишите класс `Verb` так, чтобы он хранил информацию о согласовании. Маркеры согласования находятся на конце глагола и составляют закрытый класс глосс (можете догадаться по какому паттерну они формируются). Храните либо эту глоссу, либо `None`, если согласования нет.

Для слов, для которых вы не можете определить часть речи, используйте класс `Word`.

Цель: получить на выходе список слов-объектов. Помните, что сначала вам нужно добавить эвенский язык в список доступных.